# PhotoShield — Colab runner

Runs every phase on a free GPU: protect (PGD on the SD VAE encoder + ArcFace identity + optional EOT),
evaluate against real open editors (SD 1.5 img2img, InstructPix2Pix), and the Gradio UI.

CPU laptop: 512 px / 50 steps with identity takes tens of minutes. T4: well under a minute.

**Before you start:** runtime → Change runtime type → GPU (T4).

Only use photos you own or have permission to use.

In [ ]:
!nvidia-smi -L
!pip install -q diffusers transformers accelerate safetensors lpips scikit-image onnx onnxruntime onnx2torch opencv-python-headless gradio "huggingface_hub<2"
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## 1. Get the code
Either clone your repo (if pushed) or upload a zip of the project folder.

In [ ]:
# Option A: from git (edit the URL)
# !git clone https://github.com/<you>/photoshield.git /content/photoshield

# Option B: upload a zip of the project (exclude .venv)
from google.colab import files
import zipfile, os
# up = files.upload()  # choose photoshield.zip
# zipfile.ZipFile(list(up)[0]).extractall('/content')

%cd /content/photoshield 2>/dev/null || %cd /content/photo pro
import sys; sys.path.insert(0, os.getcwd())
import photoshield; print('photoshield', photoshield.__version__, 'from', os.getcwd())

In [ ]:
# ArcFace (InsightFace buffalo_l, w600k_r50) + YuNet face detector -> weights/
import os, zipfile, urllib.request
os.makedirs('weights/buffalo_l', exist_ok=True)
if not os.path.exists('weights/buffalo_l/w600k_r50.onnx'):
    urllib.request.urlretrieve('https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip', 'buffalo_l.zip')
    zipfile.ZipFile('buffalo_l.zip').extractall('weights/buffalo_l')
if not os.path.exists('weights/face_detection_yunet_2023mar.onnx'):
    urllib.request.urlretrieve('https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx',
                               'weights/face_detection_yunet_2023mar.onnx')
print(os.listdir('weights'), os.listdir('weights/buffalo_l'))

## 2. Upload your photo
The file stays in this Colab VM. Delete it when you are done.

In [ ]:
from google.colab import files
up = files.upload()
PHOTO = list(up)[0]
print('using', PHOTO)

## 3. Protect
`eps_255=4` is the invisible setting, `8` the strongest we allow.
`w_identity=2` adds the ArcFace identity term (needs a visible face). `eot_k=4` averages over random
JPEG/resize/crop/blur views for robustness (4x slower).

In [ ]:
from photoshield.config import ProtectConfig
from photoshield.data import load_image, save_png
from photoshield.protect.pgd_engine import PGDProtector

cfg = ProtectConfig(eps=4/255, steps=100, size=512, device='cuda', w_lpips=2.0, w_identity=2.0, eot_k=0)
img = load_image(PHOTO)
prot = PGDProtector(cfg, verbose=True)
res = prot(img)
save_png(res.image, 'protected.png')
work = prot.working_image(img); save_png(work, 'original_512.png')
print('
latent', round(res.info['loss_start'],4), '->', round(res.info['loss_end'],4),
      '| L-inf', round(res.info['linf_255'],2), '/255 |', res.info['seconds'], 's')

## 4. Quality check
PSNR should be ≳38 dB, LPIPS ≲0.02 for `eps_255=4`.
If LPIPS is much higher, the perturbation is visible — do not post that file.

In [ ]:
from photoshield.metrics import metric_report, format_report
rep = metric_report(work, res.image, device='cuda', with_lpips=True)
print(format_report(rep))

import numpy as np, matplotlib.pyplot as plt
a = np.asarray(work); b = np.asarray(res.image)
amp = np.clip((b.astype(int) - a.astype(int)) * 16 + 128, 0, 255).astype(np.uint8)
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for x, im, t in zip(ax, [a, b, amp], ['original', 'protected', 'delta x16']):
    x.imshow(im); x.set_title(t); x.axis('off')
plt.show()

## 5. Evaluate against real editors (Phase 5)
Runs original / protected / random-noise control, each after none / JPEG q75 / resize 50% / crop 90%,
with the same prompt and seed. Writes `results.csv`, `results.md`, `results.json` + all edited PNGs.
First run downloads ~2.6 GB per editor (fp16).

In [ ]:
!python -m photoshield.evaluate --editor img2img --original original_512.png --protected protected.png     --prompt "portrait of the person, detailed, studio lighting" --steps 30 --strength 0.5 --out-dir results/phase5/img2img
!python -m photoshield.evaluate --editor ip2p --original original_512.png --protected protected.png     --prompt "turn this into a Studio Ghibli style painting" --steps 30 --out-dir results/phase5/ip2p
from IPython.display import Markdown, display
for e in ['img2img', 'ip2p']:
    display(Markdown(open(f'results/phase5/{e}/results.md').read()))

## 6. Gradio UI (Phase 6)
Opens a public link. Stop the cell to shut it down.

In [ ]:
from photoshield.app import build_app
build_app().launch(share=True)

## 7. Download the protected PNG

**Honest note:** read the Phase 5 table before claiming anything. A drop on SD 1.5 does not mean ChatGPT/Gemini edits fail.

In [ ]:
from google.colab import files
files.download('protected.png')